Q1. Add a new column total_amount = quantity * price using withColumn.

In [35]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
            .appName('day_02') \
            .getOrCreate()



In [36]:

read_csv = (
        spark.read \
                .option('header','true')
                .option('inferSchema','true')
                .csv('data_01.csv')
)
# print(read_csv.describe)
# read_csv.show()


''' 

from pyspark.sql.functions import col
read_csv.withColumn('total_amount', col('quantity') * col('price')).show()

'''



# total_amount = quantity * price
from pyspark.sql.functions import col
# read_csv.withColumn('total_amount' , col('quantity') * col('price'));
df2 = read_csv.withColumn('total_amount' , read_csv['quantity'] * read_csv['price']);

df2.show()



+--------+-----------+----------+--------+-------+----------+------------------+
|order_id|customer_id|   product|quantity|  price|order_date|      total_amount|
+--------+-----------+----------+--------+-------+----------+------------------+
|       1|        101|     Phone|       2| 699.99|2023-01-05|           1399.98|
|       2|        102|    Laptop|       1|1199.99|2023-01-07|           1199.99|
|       3|        103|    Tablet|       3| 449.99|2023-01-10|           1349.97|
|       4|        104|     Watch|       2| 299.99|2023-01-12|            599.98|
|       5|        105|Headphones|       4| 149.99|2023-01-15|            599.96|
|       6|        106|     Phone|       1| 799.99|2023-01-18|            799.99|
|       7|        107|    Camera|       2| 549.99|2023-01-20|           1099.98|
|       8|        108|    Laptop|       1| 999.99|2023-01-22|            999.99|
|       9|        109|   Speaker|       3| 199.99|2023-01-25|            599.97|
|      10|        110|    Ta

In [37]:

df3     =   df2.orderBy(df2.total_amount , ascending=False)

'''
# Correct
df2.orderBy(col('total_amount'), ascending=False).show(5)

# Or
df2.orderBy(col('total_amount').desc()).limit(5).show()
'''
df3.show()


+--------+-----------+-------+--------+-------+----------+------------------+
|order_id|customer_id|product|quantity|  price|order_date|      total_amount|
+--------+-----------+-------+--------+-------+----------+------------------+
|      55|        105| Laptop|       2|1219.99|2023-05-26|           2439.98|
|     200|        110| Laptop|       2|1209.99|2024-06-02|           2419.98|
|     177|        107| Laptop|       2|1199.99|2024-04-06|           2399.98|
|      70|        120| Laptop|       2|1189.99|2023-07-03|           2379.98|
|      42|        112| Laptop|       2|1179.99|2023-04-23|           2359.98|
|     151|        101| Laptop|       2|1159.99|2024-01-28|           2319.98|
|     117|        107| Laptop|       2|1149.99|2023-11-01|           2299.98|
|      98|        108| Laptop|       2|1139.99|2023-09-13|           2279.98|
|      99|        109|  Phone|       3| 759.99|2023-09-16|2279.9700000000003|
|     175|        105|  Phone|       3| 739.99|2024-04-01|2219.9

In [38]:
from pyspark.sql.functions import *

df4 = df2.groupBy('product').agg(
    max('price').alias('max_price'),
    min('price').alias('min_price'),
    avg('price').alias('avg_price')
)
df4.show()


+----------+---------+---------+------------------+
|   product|max_price|min_price|         avg_price|
+----------+---------+---------+------------------+
|   Speaker|   219.99|   169.99| 197.6685714285713|
|     Phone|   859.99|   699.99| 772.2122222222229|
|    Laptop|  1299.99|   999.99|1187.9900000000007|
|    Camera|   649.99|   499.99| 557.8471428571427|
|    Tablet|   469.99|   379.99|432.05896551724123|
|     Watch|   349.99|   259.99|310.16241379310327|
|Headphones|   164.99|   124.99|142.98999999999995|
+----------+---------+---------+------------------+



In [39]:
# Correct
filtered_df = df2.filter(col('product').rlike('(?i)phone'))
filtered_df.show()

# Cleaner alternative
filtered_df = df2.filter(lower(col('product')).contains('phone'))
filtered_df.show()

+--------+-----------+----------+--------+------+----------+------------------+
|order_id|customer_id|   product|quantity| price|order_date|      total_amount|
+--------+-----------+----------+--------+------+----------+------------------+
|       1|        101|     Phone|       2|699.99|2023-01-05|           1399.98|
|       5|        105|Headphones|       4|149.99|2023-01-15|            599.96|
|       6|        106|     Phone|       1|799.99|2023-01-18|            799.99|
|      12|        102|Headphones|       5|129.99|2023-02-05|            649.95|
|      13|        103|     Phone|       2|749.99|2023-02-08|           1499.98|
|      18|        108|     Phone|       3|699.99|2023-02-20|2099.9700000000003|
|      20|        110|Headphones|       3|159.99|2023-02-25|            479.97|
|      22|        112|     Phone|       1|849.99|2023-03-03|            849.99|
|      27|        117|Headphones|       2|139.99|2023-03-16|            279.98|
|      29|        119|     Phone|       

In [42]:
# Filter orders where product contains 'Phone' (case-insensitive) using regexp functions.


from pyspark.sql.functions import col,regexp_like


filtered_df = df2.withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))   \
                    .withColumn('year', year(col('order_date'))) \
                        .withColumn('month', month(col('order_date')))
filtered_df.printSchema()
filtered_df.show()


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+--------+-----------+----------+--------+-------+----------+------------------+----+-----+
|order_id|customer_id|   product|quantity|  price|order_date|      total_amount|year|month|
+--------+-----------+----------+--------+-------+----------+------------------+----+-----+
|       1|        101|     Phone|       2| 699.99|2023-01-05|           1399.98|2023|    1|
|       2|        102|    Laptop|       1|1199.99|2023-01-07|           1199.99|2023|    1|
|       3|        103|    Tablet|       3| 449.99|2023-01-10|           1349.97|2023|    1|
|       4|        104|     Watch|       2| 299.99|2023-01-12|            599.98|2